# Exercise 7: DataFrame API usage

In this exercise we will perform joins, filters, aggregations, window functions, expressions, broadcast, explain.

## Step 1: Example DataFrames Creation

For this example we will create 3 DataFrames with 2-3 columns, these DataFrames represen:
- users
- sales
- countries

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_users = spark.createDataFrame([
    (1, "Axel", "MX"),
    (2, "Luis", "MX"),
    (3, "Ana", "CO"),
    (4, "Maria", None),
], ["user_id", "nombre", "pais"])

df_sales = spark.createDataFrame([
    (1, "2024-01-01", 100),
    (1, "2024-01-05", 250),
    (2, "2024-01-03", 300),
    (5, "2024-01-02", 500),
], ["user_id", "fecha", "monto"])

df_countries = spark.createDataFrame([
    ("MX", "México"),
    ("CO", "Colombia"),
], ["pais", "pais_nombre"])

## Advanced Joins

In [0]:
# Left Joins (users with sales)
df_left = df_users.join(df_sales, "user_id", "left")
df_left.show()

+-------+------+----+----------+-----+
|user_id|nombre|pais|     fecha|monto|
+-------+------+----+----------+-----+
|      1|  Axel|  MX|2024-01-05|  250|
|      1|  Axel|  MX|2024-01-01|  100|
|      2|  Luis|  MX|2024-01-03|  300|
|      3|   Ana|  CO|      NULL| NULL|
|      4| Maria|NULL|      NULL| NULL|
+-------+------+----+----------+-----+



In [0]:
# Full Outer Join (To see who is missing on each side)
df_full = df_users.join(df_sales, "user_id", "full")
df_full.show()

+-------+------+----+----------+-----+
|user_id|nombre|pais|     fecha|monto|
+-------+------+----+----------+-----+
|      1|  Axel|  MX|2024-01-05|  250|
|      2|  Luis|  MX|2024-01-03|  300|
|      3|   Ana|  CO|      NULL| NULL|
|      4| Maria|NULL|      NULL| NULL|
|      1|  Axel|  MX|2024-01-01|  100|
|      5|  NULL|NULL|2024-01-02|  500|
+-------+------+----+----------+-----+



In [0]:
# Join with Broadcast (Optimization)
df_opt = df_left.join(F.broadcast(df_countries), "pais", "left")
df_opt.show()

+----+-------+------+----------+-----+-----------+
|pais|user_id|nombre|     fecha|monto|pais_nombre|
+----+-------+------+----------+-----+-----------+
|  MX|      1|  Axel|2024-01-05|  250|     México|
|  MX|      1|  Axel|2024-01-01|  100|     México|
|  MX|      2|  Luis|2024-01-03|  300|     México|
|  CO|      3|   Ana|      NULL| NULL|   Colombia|
|NULL|      4| Maria|      NULL| NULL|       NULL|
+----+-------+------+----------+-----+-----------+



## Advanced Filters

In [0]:
df_filtrado = df_opt.filter(
    (F.col("monto") > 100) |
    (F.col("pais").isNull())
)
df_filtrado.show()


+----+-------+------+----------+-----+-----------+
|pais|user_id|nombre|     fecha|monto|pais_nombre|
+----+-------+------+----------+-----+-----------+
|  MX|      1|  Axel|2024-01-05|  250|     México|
|  MX|      2|  Luis|2024-01-03|  300|     México|
|NULL|      4| Maria|      NULL| NULL|       NULL|
+----+-------+------+----------+-----+-----------+



## Real Aggregations

In [0]:
## Sales Total by User
df_agg = df_opt.groupBy("user_id").agg(
    F.count("*").alias("num_transacciones"),
    F.sum("monto").alias("total_ventas"),
    F.avg("monto").alias("promedio_venta")
)
df_agg.show()


+-------+-----------------+------------+--------------+
|user_id|num_transacciones|total_ventas|promedio_venta|
+-------+-----------------+------------+--------------+
|      1|                2|         350|         175.0|
|      2|                1|         300|         300.0|
|      3|                1|        NULL|          NULL|
|      4|                1|        NULL|          NULL|
+-------+-----------------+------------+--------------+



## Window Functions

In [0]:
# Sales Ranking by User
window_user = Window.partitionBy("user_id").orderBy(F.col("monto").desc())

df_window = df_opt.withColumn(
    "rank_venta", F.rank().over(window_user)
)
df_window.show()


+----+-------+------+----------+-----+-----------+----------+
|pais|user_id|nombre|     fecha|monto|pais_nombre|rank_venta|
+----+-------+------+----------+-----+-----------+----------+
|  MX|      1|  Axel|2024-01-05|  250|     México|         1|
|  MX|      1|  Axel|2024-01-01|  100|     México|         2|
|  MX|      2|  Luis|2024-01-03|  300|     México|         1|
|  CO|      3|   Ana|      NULL| NULL|   Colombia|         1|
|NULL|      4| Maria|      NULL| NULL|       NULL|         1|
+----+-------+------+----------+-----+-----------+----------+



## Column Expressions

In [0]:
#Sales Classifications
df_clasificado = df_opt.withColumn(
    "categoria",
    F.when(F.col("monto") >= 300, "ALTA")
     .when(F.col("monto") >= 150, "MEDIA")
     .otherwise("BAJA")
)
df_clasificado.show()

+----+-------+------+----------+-----+-----------+---------+
|pais|user_id|nombre|     fecha|monto|pais_nombre|categoria|
+----+-------+------+----------+-----+-----------+---------+
|  MX|      1|  Axel|2024-01-05|  250|     México|    MEDIA|
|  MX|      1|  Axel|2024-01-01|  100|     México|     BAJA|
|  MX|      2|  Luis|2024-01-03|  300|     México|     ALTA|
|  CO|      3|   Ana|      NULL| NULL|   Colombia|     BAJA|
|NULL|      4| Maria|      NULL| NULL|       NULL|     BAJA|
+----+-------+------+----------+-----+-----------+---------+



## Null Management

In [0]:
df_nulls = df_opt.fillna({"pais": "DESCONOCIDO"})
df_nulls.show()


+-----------+-------+------+----------+-----+-----------+
|       pais|user_id|nombre|     fecha|monto|pais_nombre|
+-----------+-------+------+----------+-----+-----------+
|         MX|      1|  Axel|2024-01-05|  250|     México|
|         MX|      1|  Axel|2024-01-01|  100|     México|
|         MX|      2|  Luis|2024-01-03|  300|     México|
|         CO|      3|   Ana|      NULL| NULL|   Colombia|
|DESCONOCIDO|      4| Maria|      NULL| NULL|       NULL|
+-----------+-------+------+----------+-----+-----------+



## Physical plan (Catalyst + Tungsten)

In [0]:
df_opt.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(LeftOuter, [pais])
:- 'Join UsingJoin(LeftOuter, [user_id])
:  :- 'UnresolvedSubqueryColumnAliases [user_id, nombre, pais]
:  :  +- LocalRelation [user_id#11117L, nombre#11118, pais#11119]
:  +- 'UnresolvedSubqueryColumnAliases [user_id, fecha, monto]
:     +- LocalRelation [user_id#11133L, fecha#11134, monto#11135L]
+- 'UnresolvedHint broadcast
   +- 'UnresolvedSubqueryColumnAliases [pais, pais_nombre]
      +- LocalRelation [pais#11146, pais_nombre#11147]

== Analyzed Logical Plan ==
pais: string, user_id: bigint, nombre: string, fecha: string, monto: bigint, pais_nombre: string
Project [pais#11546, user_id#11544L, nombre#11545, fecha#11548, monto#11549L, pais_nombre#11551]
+- Join LeftOuter, (pais#11546 = pais#11550)
   :- Project [user_id#11544L, nombre#11545, pais#11546, fecha#11548, monto#11549L]
   :  +- Join LeftOuter, (user_id#11544L = user_id#11547L)
   :     :- Project [user_id#11117L AS user_id#11544L, nombre#11118 AS nombre#11545, 